# 🤖 MBTI Personality Detection — RAG + LLM Agent (LangChain)

---

**Architecture: Agent = LLM + Planning + Tool Use + Loop + Memory**

| Component | Implementation |
|-----------|---------------|
| **Framework** | **LangChain** — Agent, Tools, VectorStore, Embeddings, LLM integration |
| **LLM** | Qwen2.5-7B-Instruct (4-bit quantization) via `HuggingFacePipeline` |
| **Planning** | LangChain `create_react_agent` — ReAct prompt-based **autonomous** reasoning |
| **Tool Use** | Hybrid: pre-computed context (Python) + LangChain `@tool` for reasoning-dependent tools |
| **Loop** | `AgentExecutor` with configurable `max_iterations` — iterations freed for actual reasoning |
| **Memory** | Short-term: `AgentExecutor` conversation history \| Long-term: FAISS experience store |
| **RAG** | 3-stage pipeline: FAISS bi-encoder → Cross-encoder reranking → Lost-in-the-middle reordering |
| **Guardrail** | SLM Judge — independent Qwen2.5-3B-Instruct does binary ACCEPT/REJECT per dimension |

**Why Hybrid Architecture (not full pipeline or full agent)?**
- **Full pipeline**: No reasoning, fixed steps, can't adapt to ambiguous cases
- **Full agent**: Wastes 2/3 iterations on deterministic ops (text features + similar post retrieval)
- **Hybrid**: Deterministic steps pre-computed in Python → injected into prompt. Agent iterations freed for:
  - Retrieving MBTI knowledge when a specific dimension is uncertain
  - Checking memory for similar past classifications
  - Revising prediction after Judge rejection

**RAG Pipeline (3-stage retrieval):**
- **Embedding**: `BAAI/bge-small-en-v1.5` (384d) — better semantic quality than MiniLM for personality analysis
- **Stage 1 — FAISS Bi-Encoder**: Fast approximate search → top-15 candidates
- **Stage 2 — Cross-Encoder Reranking**: `ms-marco-MiniLM-L-6-v2` reranks by token-level cross-attention → top-5. Filters out posts with similar vocabulary but different MBTI types
- **Stage 3 — Lost-in-the-Middle Reordering**: Best results at start+end positions where LLMs attend most
- **VectorStore 1 — Similar Posts**: Training posts → few-shot examples (with reranking)
- **VectorStore 2 — MBTI Knowledge Base**: Dimension-tagged chunks → filtered retrieval by dimension

**SLM Judge Guardrail (Hậu kiểm — Asymmetric Architecture):**
- **Generator** (Qwen2.5-7B): Full ReAct agent with RAG + tools + reasoning
- **Judge** (Qwen2.5-3B): Lightweight independent binary classifier per dimension
- Judge receives posts + predicted trait + dimension descriptions → outputs `{"verdict": "ACCEPT"|"REJECT", "reason": "..."}`
- Rejected dimensions trigger agent revision with actionable feedback
- After `MAX_GUARDRAIL_RETRIES` rejections, force-accept to prevent infinite loops

**LangChain Components Used:**
- `langchain_huggingface.HuggingFaceEmbeddings` — embedding model wrapper
- `langchain_huggingface.HuggingFacePipeline` — LLM wrapper
- `langchain_community.vectorstores.FAISS` — vector store + retriever
- `langchain.tools.tool` — tool decorator with auto JSON schema
- `langchain.agents.create_react_agent` — ReAct agent factory
- `langchain.agents.AgentExecutor` — agent execution loop with memory

**Datasets supported:**
- MBTI-500 (~106K rows): `/kaggle/input/.../MBTI 500.csv`
- Kaggle MBTI (~8600 rows): `/kaggle/input/.../mbti_1.csv`

## 📦 Cell 1 — Install Dependencies

In [ ]:
# ⚠️ Do NOT install/upgrade torch or transformers — Kaggle has CUDA 12-compatible versions pre-installed.
!pip install -q --no-deps bitsandbytes accelerate
!pip install -q faiss-cpu
!pip install -q "sentence-transformers>=3.0,<4.0"
!pip install -q "langchain>=0.3.0,<0.4.0" "langchain-community>=0.3.0,<0.4.0" "langchain-core>=0.3.0,<0.4.0" langchain-huggingface

# ── Verify torch was NOT upgraded to CUDA 13 ──
import subprocess, sys
_tv = subprocess.check_output([sys.executable, '-c',
    'import torch; print(torch.__version__)']).decode().strip()
print(f'✅ torch version: {_tv}')
if 'cu130' in _tv or 'cu131' in _tv:
    print('❌ ERROR: torch was upgraded to CUDA 13 but Kaggle only has CUDA 12!')
    print('   → Go to Runtime > Restart session, then re-run ALL cells.')
    raise RuntimeError(f'torch {_tv} requires CUDA 13. Restart kernel to restore Kaggle defaults.')

## 📥 Cell 2 — Imports

In [ ]:
import os, re, json, time, warnings, gc
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import faiss

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import pipeline as hf_pipeline

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    classification_report
)
from sklearn.preprocessing import label_binarize

# ── LangChain imports ──
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS as LangChainFAISS
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_react_agent, AgentExecutor

import langchain
print(f'✅ Imports done (LangChain {langchain.__version__} + HuggingFace)')

warnings.filterwarnings('ignore')

# Suppress HuggingFace "using pipelines sequentially on GPU" warning
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
import logging
logging.getLogger('transformers.pipelines').setLevel(logging.ERROR)

## ⚙️ Cell 3 — Config

In [ ]:
# ───────────────────────────────────────────────────────────
# CONFIG — all paths and hyperparameters here
# ───────────────────────────────────────────────────────────
CONFIG = {
    # Paths
    'DATA_PATH_500':  '/kaggle/input/datasets/zeyadkhalid/mbti-personality-types-500-dataset/MBTI 500.csv',
    'DATA_PATH_8K':   '/kaggle/input/datasets/datasnaek/mbti-type/mbti_1.csv',
    'RESULT_DIR':     '/kaggle/working/results',

    # Data preprocessing
    'MAX_POSTS':      50,
    'MAX_WORDS':      70,

    # RAG — Retrieval
    'RAG_TOP_K':        3,                            # final results after reranking (speed-optimized)
    'RAG_RETRIEVE_K':   10,                           # FAISS candidates before reranking (speed-optimized)
    'EMBED_MODEL':      'BAAI/bge-small-en-v1.5',    # upgraded: better semantic quality, same 384d
    'EMBED_DIM':        384,

    # RAG — Cross-Encoder Reranking
    # After FAISS retrieves RAG_RETRIEVE_K candidates by embedding similarity,
    # a cross-encoder reranks them by deeper token-level attention.
    # This filters out posts with similar vocabulary but different MBTI types.
    'RERANKER_ENABLED': True,
    'RERANKER_MODEL':   'cross-encoder/ms-marco-MiniLM-L-6-v2',

    # Agent (Generator)
    'LLM_MODEL':      'Qwen/Qwen2.5-7B-Instruct',
    'MAX_AGENT_ITERATIONS': 3,

    # Long-term Memory
    'MEMORY_CONF_THRESHOLD': 0.80,

    # ── SLM Judge Guardrail (Asymmetric Architecture) ──
    # Generator = Qwen2.5-7B (heavyweight, does RAG + reasoning)
    # Judge     = Qwen2.5-3B (lightweight, only does binary ACCEPT/REJECT)
    #
    # The Judge reviews each MBTI dimension independently:
    #   - Receives: posts excerpt + predicted letter + dimension knowledge
    #   - Outputs: structured JSON {verdict: ACCEPT/REJECT, reason: ...}
    #   - Runs on CPU or shares GPU — ~3B params ≈ <2GB VRAM in 4-bit
    'GUARDRAIL_ENABLED':        True,
    'JUDGE_MODEL':              'Qwen/Qwen2.5-3B-Instruct',
    'JUDGE_MAX_NEW_TOKENS':     100,      # short structured JSON output
    'JUDGE_PREMISE_MAX_CHARS':  600,      # max chars from posts for judge context
    'MAX_GUARDRAIL_RETRIES':    2,        # max rejections before forced accept
    'JUDGE_SELECTIVE_ENABLED':  True,     # only judge low-confidence dims on first attempt
    'JUDGE_CONF_THRESHOLD':     0.75,     # judge dims with conf < threshold
}

os.makedirs(CONFIG['RESULT_DIR'], exist_ok=True)

# ── GPU detection ──
NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda' if NUM_GPUS > 0 else 'cpu')
print(f'🖥  GPUs available: {NUM_GPUS}  |  Device: {DEVICE}')
print(f'🛡️  Guardrail:     {"ON — SLM Judge (Qwen2.5-3B)" if CONFIG["GUARDRAIL_ENABLED"] else "OFF — no verification"}')
print(f'🔍  Embedding:     {CONFIG["EMBED_MODEL"]} ({CONFIG["EMBED_DIM"]}d)')
print(f'🔄  Reranker:      {"ON — " + CONFIG["RERANKER_MODEL"] if CONFIG["RERANKER_ENABLED"] else "OFF"}')
print(f'   FAISS top-{CONFIG["RAG_RETRIEVE_K"]} → Rerank → top-{CONFIG["RAG_TOP_K"]}')

# ── Constants ──
MBTI_TYPES  = ['infj','infp','intj','intp','isfj','isfp','istj','istp',
               'enfj','enfp','entj','entp','esfj','esfp','estj','estp']
MBTI_EXTRA  = ['introvert','extrovert','introverted','extroverted',
               'sensing','intuition','thinking','feeling','judging','perceiving']
ALL_MASK    = MBTI_TYPES + MBTI_EXTRA
MASK_RE     = re.compile(r'\b(' + '|'.join(ALL_MASK) + r')\b', re.IGNORECASE)

LABEL_COL   = 'type'
TEXT_COL    = 'posts'
LABEL_COLS  = ['label_ie', 'label_sn', 'label_tf', 'label_jp']
DIM_NAMES   = ['I/E', 'S/N', 'T/F', 'J/P']

## 📂 Cell 4 — Load & Preprocess Dataset

In [ ]:
# ── Load dataset (prefer 500K, fall back to 8K) ──
def load_dataset(cfg):
    for path in [cfg['DATA_PATH_500'], cfg['DATA_PATH_8K']]:
        try:
            df = pd.read_csv(path)
            print(f'✅ Loaded: {path}  shape={df.shape}')
            return df[:1300]
        except FileNotFoundError:
            continue
    raise FileNotFoundError('❌ No dataset found. Add data and check paths.')

df_raw = load_dataset(CONFIG)


# ── Step 1: Map 16-class MBTI → 4 binary labels ──
def map_mbti_to_binary(mbti_str):
    m = mbti_str.strip().upper()
    return int(m[0]=='I'), int(m[1]=='N'), int(m[2]=='F'), int(m[3]=='J')

df_raw[['label_ie','label_sn','label_tf','label_jp']] = (
    df_raw[LABEL_COL].apply(lambda x: pd.Series(map_mbti_to_binary(x)))
)


# ── Step 2: Psycholinguistic masking + truncation ──
def preprocess_user_posts(raw_str, max_posts=CONFIG['MAX_POSTS'], max_words=CONFIG['MAX_WORDS']):
    posts = raw_str.split('|||')
    result = []
    for p in posts[:max_posts]:
        p = MASK_RE.sub('<type>', p.strip())
        p = ' '.join(p.split()[:max_words])
        if p:
            result.append(p)
    return result

print('⏳ Preprocessing posts (masking + truncation)...')
df_raw['processed_posts'] = df_raw[TEXT_COL].apply(preprocess_user_posts)
df_raw['concat_posts'] = df_raw['processed_posts'].apply(lambda x: ' ||| '.join(x))
print(f'✅ Preprocessing done. Rows: {len(df_raw)}')


# ── Step 3: Stratified split across ALL 4 axes jointly ──
df_raw['combo_label'] = df_raw[LABEL_COLS].astype(str).agg(''.join, axis=1)

train_df, temp_df = train_test_split(
    df_raw, test_size=0.4, stratify=df_raw['combo_label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['combo_label'], random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'\n📊 Split → Train:{len(train_df)} | Val:{len(val_df)} | Test:{len(test_df)}')

## 📚 Cell 5 — MBTI Knowledge Base

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  MBTI KNOWLEDGE BASE (Domain Knowledge for RAG)
# ═══════════════════════════════════════════════════════════════

MBTI_DIM_KNOWLEDGE = {
    "I/E": (
        "Introversion (I) indicators in text: reflective and introspective language, longer and more "
        "complex sentences, frequent use of 'I think', 'in my opinion', hedging words like 'perhaps' "
        "or 'maybe', discussion of inner thoughts and ideas, fewer social references, preference for "
        "deep topics over small talk, lower frequency of exclamation marks, references to solitude and "
        "personal space, self-referential language, analytical introspection.\n"
        "Extraversion (E) indicators in text: action-oriented and energetic language, shorter direct "
        "sentences, frequent exclamation marks, references to social activities and groups, use of "
        "'we/us/our', discussion of events and people, more casual and conversational tone, references "
        "to parties, gatherings, collaboration, social energy, group dynamics."
    ),
    "S/N": (
        "Sensing (S) indicators in text: concrete and specific language, references to tangible details "
        "and facts, step-by-step descriptions, present-focused or past-experience based writing, "
        "practical vocabulary, sensory words (see, hear, feel, touch), specific numbers and dates, "
        "discussion of real-world events, hands-on experiences, routine references.\n"
        "Intuition (N) indicators in text: abstract and theoretical language, metaphors and analogies, "
        "discussion of possibilities and 'what if' scenarios, future-oriented thinking, conceptual "
        "vocabulary, references to patterns and connections, philosophical discussions, creative and "
        "imaginative expressions, symbolic language, big-picture focus."
    ),
    "T/F": (
        "Thinking (T) indicators in text: logical and analytical language, cause-effect reasoning "
        "using words like 'because', 'therefore', 'logically', objective and impersonal tone, "
        "critique and debate style, fewer emotional expressions, systematic analysis, focus on truth "
        "and accuracy, direct disagreement, technical vocabulary, problem-solving focus.\n"
        "Feeling (F) indicators in text: emotional and empathetic language, personal values expression, "
        "use of feeling words ('feel', 'love', 'care', 'heart'), harmony-seeking communication, "
        "people-focused discussions, supportive and encouraging tone, references to personal impact, "
        "moral judgments based on values rather than logic, relationship-oriented content."
    ),
    "J/P": (
        "Judging (J) indicators in text: decisive and definitive language using 'should', 'must', "
        "'will', structured and organized writing, closure-seeking statements, references to plans, "
        "schedules, and goals, preference for order, list-making tendency, strong opinions stated as "
        "facts, completion-focused language, methodical approach.\n"
        "Perceiving (P) indicators in text: open-ended and exploratory language using 'maybe', "
        "'might', 'what about', flexible and adaptive writing, multiple options considered, "
        "spontaneous topic changes, casual attitude toward deadlines, curiosity-driven exploration, "
        "'going with the flow' references, incomplete thoughts, tangential thinking."
    ),
}

MBTI_TYPE_PROFILES = {
    "INTJ": "Ni-Te-Fi-Se. Strategic, independent, determined. Precise analytical language, long-term vision, minimal small talk, direct communication, confident assertions, systems thinking, rare emotional expression, future-planning focus.",
    "INTP": "Ti-Ne-Si-Fe. Analytical, curious, innovative. Complex sentence structures, explores multiple angles, theoretical vocabulary, questions everything, logical precision, detached tone, idea-driven, loves intellectual debate.",
    "ENTJ": "Te-Ni-Se-Fi. Decisive, ambitious, commanding. Direct and authoritative language, efficiency-focused, goal-oriented, structured arguments, leadership references, action-oriented planning, strategic communication.",
    "ENTP": "Ne-Ti-Fe-Si. Quick-witted, argumentative, inventive. Playful language, devil's advocate positions, humor and sarcasm, idea-jumping, debate-seeking, creative connections, challenges assumptions, energetic discourse.",
    "INFJ": "Ni-Fe-Ti-Se. Insightful, principled, compassionate. Metaphorical language, deep reflections, idealistic expressions, future-oriented, empathetic but private, symbolic thinking, meaningful connections, visionary language.",
    "INFP": "Fi-Ne-Si-Te. Idealistic, empathetic, creative. Poetic and personal language, value-driven expressions, emotional depth, authenticity focus, imaginative descriptions, identity exploration, artistic sensibility.",
    "ENFJ": "Fe-Ni-Se-Ti. Charismatic, inspiring, empathetic. Encouraging and motivational language, people-focused, visionary statements, warm tone, community references, natural leadership, harmony-building communication.",
    "ENFP": "Ne-Fi-Te-Si. Enthusiastic, creative, sociable. Energetic language, many exclamations, rapid topic shifts, passionate expressions, optimistic tone, brainstorming style, explores possibilities, emotionally expressive.",
    "ISTJ": "Si-Te-Fi-Ne. Practical, reliable, methodical. Structured factual language, duty references, traditional values, step-by-step descriptions, concrete details, rule-following, historical references.",
    "ISFJ": "Si-Fe-Ti-Ne. Caring, reliable, observant. Supportive language, detail-oriented descriptions, tradition-respecting, nurturing tone, practical helpfulness, loyalty-focused, modest expression.",
    "ESTJ": "Te-Si-Ne-Fi. Organized, direct, traditional. Clear directives, fact-based arguments, efficiency-focused, authoritative tone, rule-following references, practical leadership.",
    "ESFJ": "Fe-Si-Ne-Ti. Caring, sociable, traditional. Warm and inclusive language, social harmony focus, community references, supportive expressions, practical care, people-oriented.",
    "ISTP": "Ti-Se-Ni-Fe. Practical, observant, analytical. Concise factual language, action-focused, problem-solving oriented, minimal emotional expression, hands-on references, independent thinking.",
    "ISFP": "Fi-Se-Ni-Te. Gentle, sensitive, artistic. Aesthetic language, present-moment focus, personal values, sensory descriptions, quiet emotional depth, creative expression.",
    "ESTP": "Se-Ti-Fe-Ni. Bold, direct, perceptive. Action-oriented short sentences, present-focused, practical and energetic, humor, risk-taking references, competitive language.",
    "ESFP": "Se-Fi-Te-Ni. Spontaneous, energetic, playful. Vivid sensory language, social engagement, fun-focused, enthusiastic tone, living-in-the-moment expressions, entertaining style.",
}

print(f'✅ Knowledge base ready: {len(MBTI_DIM_KNOWLEDGE)} dimension descriptions + {len(MBTI_TYPE_PROFILES)} type profiles')

## 🔍 Cell 6 — Build LangChain FAISS VectorStores + Cross-Encoder Reranker

**RAG Retrieval Pipeline** (3-stage):
1. **FAISS Bi-Encoder** — fast approximate search over all documents → top-15 candidates
2. **Cross-Encoder Reranking** — `ms-marco-MiniLM-L-6-v2` reranks by token-level attention → top-5
3. **Lost-in-the-Middle Reordering** — best results placed at start+end of context (where LLMs attend most)

**Why Reranking?**
- FAISS (bi-encoder) matches by surface-level embedding similarity → can return posts with similar vocabulary but different MBTI types
- Cross-encoder does token-by-token cross-attention between query and document → catches semantic differences that bi-encoder misses

**Dimension-Tagged Knowledge Base:**
- Each knowledge chunk has explicit `dimension` metadata (`I/E`, `S/N`, `T/F`, `J/P`, or `all`)
- When agent queries about a specific dimension, metadata filtering returns only relevant knowledge

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  BUILD LANGCHAIN FAISS VECTORSTORES + CROSS-ENCODER RERANKER
# ═══════════════════════════════════════════════════════════════

# ── LangChain Embeddings (wraps sentence-transformers) ──
print(f'⏳ Loading embeddings: {CONFIG["EMBED_MODEL"]}...')
lc_embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG['EMBED_MODEL'],
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

# ── Cross-Encoder Reranker ──
# After FAISS retrieves top-K candidates by embedding similarity (bi-encoder),
# the cross-encoder reranks them using token-level cross-attention.
# This catches cases where posts share vocabulary but differ in MBTI type.
reranker = None
if CONFIG['RERANKER_ENABLED']:
    try:
        from sentence_transformers import CrossEncoder
    except ImportError:
        from sentence_transformers.cross_encoder import CrossEncoder
    print(f'⏳ Loading reranker: {CONFIG["RERANKER_MODEL"]}...')
    reranker = CrossEncoder(CONFIG['RERANKER_MODEL'], device='cpu')
    print(f'✅ Reranker loaded: {CONFIG["RERANKER_MODEL"]}')


def rerank_docs(query: str, docs_with_scores: list, top_k: int) -> list:
    """Rerank FAISS results using cross-encoder, then apply lost-in-the-middle reordering.

    Lost-in-the-middle: LLMs attend best to the START and END of context.
    After reranking, we interleave: [rank1, rank3, rank5, ..., rank4, rank2]
    so the most relevant docs are at positions 1 (start) and N (end).
    """
    if reranker is None or len(docs_with_scores) <= top_k:
        return docs_with_scores[:top_k]

    # Cross-encoder scoring: pairs of (query, document)
    pairs = [(query[:500], doc.page_content[:500]) for doc, _ in docs_with_scores]
    ce_scores = reranker.predict(pairs)

    # Sort by cross-encoder score (descending)
    scored = sorted(
        zip(docs_with_scores, ce_scores),
        key=lambda x: x[1], reverse=True
    )
    reranked = [doc_score for doc_score, _ in scored[:top_k]]

    # ── Lost-in-the-middle reordering ──
    # Place rank 1,3,5... at start, rank 2,4,6... at end (reversed)
    # Result: best items at first and last positions
    if len(reranked) >= 3:
        odd_pos = reranked[0::2]   # rank 1, 3, 5, ...
        even_pos = reranked[1::2]  # rank 2, 4, 6, ...
        reranked = odd_pos + even_pos[::-1]

    return reranked


# ── VectorStore 1: Training posts → similar-post retrieval (few-shot) ──
print('⏳ Building LangChain FAISS vectorstore for training posts...')
post_documents = [
    Document(
        page_content=row['concat_posts'],
        metadata={'type': row[LABEL_COL], 'index': i}
    )
    for i, row in train_df.iterrows()
]

# Build in batches to avoid memory issues
FAISS_BATCH = 512
post_vectorstore = LangChainFAISS.from_documents(post_documents[:FAISS_BATCH], lc_embeddings)
for start in range(FAISS_BATCH, len(post_documents), FAISS_BATCH):
    batch = post_documents[start:start + FAISS_BATCH]
    post_vectorstore.add_documents(batch)
    if (start // FAISS_BATCH) % 20 == 0:
        print(f'  ... {start + len(batch)}/{len(post_documents)} posts indexed')

post_retriever = post_vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': CONFIG['RAG_TOP_K']}
)
print(f'✅ Post vectorstore: {len(post_documents)} documents → LangChain FAISS retriever')


# ── VectorStore 2: MBTI Knowledge Base ──
# Each chunk gets explicit dimension metadata for filtered retrieval.
# When the agent queries about "T/F", metadata filtering returns only T/F knowledge
# instead of all 4 dimensions — reducing noise for the 7B model.
print('⏳ Building LangChain FAISS vectorstore for MBTI knowledge...')
knowledge_documents = []
for dim_name, text in MBTI_DIM_KNOWLEDGE.items():
    knowledge_documents.append(Document(
        page_content=text,
        metadata={
            'label': dim_name,
            'doc_type': 'dimension',
            'dimension': dim_name,  # explicit dimension tag for filtered retrieval
        }
    ))
for type_name, profile in MBTI_TYPE_PROFILES.items():
    # Type profiles contain info about all 4 dimensions → tag with all
    knowledge_documents.append(Document(
        page_content=profile,
        metadata={
            'label': type_name,
            'doc_type': 'profile',
            'dimension': 'all',
        }
    ))

knowledge_vectorstore = LangChainFAISS.from_documents(knowledge_documents, lc_embeddings)
knowledge_retriever = knowledge_vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)
print(f'✅ Knowledge vectorstore: {len(knowledge_documents)} chunks → LangChain FAISS retriever')


# ── Pre-embed test posts (use underlying model for speed) ──
print('⏳ Encoding test posts...')
from sentence_transformers import SentenceTransformer
_st_model = SentenceTransformer(CONFIG['EMBED_MODEL'], device='cpu')
test_texts_for_rag = test_df['concat_posts'].tolist()
test_embeddings_np = _st_model.encode(
    test_texts_for_rag, batch_size=128, show_progress_bar=True,
    normalize_embeddings=True
).astype(np.float32)
print(f'✅ Test embeddings ready: {test_embeddings_np.shape}')

# ── Summary ──
print(f'\n📋 RAG Pipeline Summary:')
print(f'   Embedding:  {CONFIG["EMBED_MODEL"]} ({CONFIG["EMBED_DIM"]}d)')
print(f'   Reranker:   {"ON — " + CONFIG["RERANKER_MODEL"] if CONFIG["RERANKER_ENABLED"] else "OFF"}')
print(f'   Retrieval:  FAISS top-{CONFIG["RAG_RETRIEVE_K"]} → Rerank → top-{CONFIG["RAG_TOP_K"]}')
print(f'   Reordering: Lost-in-the-middle (important info at start+end)')
print(f'   Knowledge:  Dimension-tagged metadata for filtered retrieval')

## 🧠 Cell 7 — Load LLM via LangChain HuggingFacePipeline

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  LOAD LLM via LangChain HuggingFacePipeline
# ═══════════════════════════════════════════════════════════════

# ── Pre-flight CUDA check ──
_torch_cuda = torch.__version__
print(f'🔍 torch={_torch_cuda}, CUDA={torch.version.cuda}')
if 'cu130' in _torch_cuda or 'cu131' in _torch_cuda:
    raise RuntimeError(
        f'torch {_torch_cuda} needs CUDA 13 but Kaggle has CUDA 12. '
        f'Restart kernel (Runtime > Restart session) and re-run all cells.'
    )

LLM_MODEL_NAME = CONFIG['LLM_MODEL']

print(f'⏳ Loading {LLM_MODEL_NAME} (4-bit quantization)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME, trust_remote_code=True)
llm_model_raw = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

hf_pipe = hf_pipeline(
    'text-generation',
    model=llm_model_raw,
    tokenizer=llm_tokenizer,
    max_new_tokens=256,             # ← needs room for Thought + Action + Action Input
    do_sample=False,                # ← greedy decoding = faster, deterministic
    return_full_text=False,
)

# ── Wrap as LangChain LLM ──
# NOTE: create_react_agent adds stop=["\nObservation:"] by default,
# which HuggingFacePipeline handles via post-processing (strips text at stop token).
# Combined with RobustReActParser (Cell 8), this prevents hallucinated tool outputs.
llm = HuggingFacePipeline(pipeline=hf_pipe)

print(f'✅ LLM loaded: {LLM_MODEL_NAME}  |  dtype=4bit-nf4  |  max_new_tokens=256')
print(f'   LangChain wrapper: HuggingFacePipeline')
print(f'   Decoding: greedy (do_sample=False) — faster than sampling')
print(f'   ReAct stop: LangChain built-in stop=[\\nObservation:] + RobustReActParser')

## 🛡️ Cell 7b — SLM Judge Guardrail (Asymmetric Architecture)

**Asymmetric Guardrail Design** — The Generator (heavy) and Judge (light) are separate models with different roles:

| Role | Model | Params | Purpose |
|------|-------|--------|---------|
| **Generator** | Qwen2.5-7B-Instruct | ~7B (4-bit) | Full ReAct agent: RAG + tool use + reasoning |
| **Judge** | Qwen2.5-3B-Instruct | ~3B (4-bit) | Binary classifier: ACCEPT/REJECT per dimension |

**Why Asymmetric?**
- Generator needs a large model for complex reasoning (tool chaining, RAG, multi-step analysis)
- Judge only does binary classification → a 3B model is sufficient and **much faster**
- No self-evaluation bias — Judge is a completely separate model
- ~3B params ≈ <1GB VRAM in 4-bit → minimal resource overhead

**How it works (Hậu kiểm — Post-Verification):**
1. Agent (7B) predicts MBTI type via ReAct loop
2. For each of 4 dimensions, the Judge (3B) receives:
   - Posts excerpt (truncated)
   - The predicted letter + dimension description from knowledge base
3. Judge outputs structured JSON: `{"dimension": "T/F", "verdict": "REJECT", "reason": "..."}`
4. If any dimension is REJECTED → agent must revise that dimension
5. After `MAX_GUARDRAIL_RETRIES` rejections → force-accept

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  SLM JUDGE GUARDRAIL (Asymmetric Architecture)
# ═══════════════════════════════════════════════════════════════
# Generator (Qwen2.5-7B) does the heavy lifting: RAG, tool use, reasoning.
# Judge (Qwen2.5-3B) is a lightweight binary classifier that reviews
# each MBTI dimension independently: ACCEPT or REJECT.
#
# The Judge receives:
#   - Posts excerpt (truncated to JUDGE_PREMISE_MAX_CHARS)
#   - Predicted letter + trait description (ONE trait only, no alternative)
# And is asked a simple YES/NO question:
#   "Does this person's writing show [trait] traits? Answer YES or NO."
#
# Why YES/NO instead of JSON?
#   - 3B models are unreliable at producing structured JSON
#   - YES/NO is a simpler classification task the model can handle
#   - The fallback defaults to ACCEPT (generator already did heavy RAG work)
#
# Controlled by CONFIG['GUARDRAIL_ENABLED']:
#   True  → load Judge model, verify predictions
#   False → skip verification entirely
# ═══════════════════════════════════════════════════════════════

_OPPOSITE = {'I': 'E', 'E': 'I', 'S': 'N', 'N': 'S', 'T': 'F', 'F': 'T', 'J': 'P', 'P': 'J'}

# ── Dimension descriptions for Judge context ──
JUDGE_DIM_DESCRIPTIONS = {
    'ie': {
        'I': "Introversion: reflective, introspective, prefers solitude, 'I think', hedging words, fewer social references, deep topics, self-referential.",
        'E': "Extraversion: energetic, social references, 'we/us', exclamation marks, group activities, casual/conversational, action-oriented.",
    },
    'sn': {
        'S': "Sensing: concrete details, facts, step-by-step, present/past-focused, sensory words, specific numbers, practical, hands-on.",
        'N': "Intuition: abstract, metaphors, 'what if', future-oriented, patterns, philosophical, creative/imaginative, big-picture.",
    },
    'tf': {
        'T': "Thinking: logical, 'because/therefore', objective, critique/debate, fewer emotions, systematic analysis, truth-focused, technical.",
        'F': "Feeling: emotional, 'feel/love/care', empathetic, harmony-seeking, people-focused, supportive, values-based, relationship-oriented.",
    },
    'jp': {
        'J': "Judging: decisive, 'should/must/will', structured, plans/schedules/goals, order, list-making, strong opinions, methodical.",
        'P': "Perceiving: open-ended, 'maybe/might', flexible, spontaneous topic changes, casual deadlines, curiosity-driven, 'going with the flow'.",
    },
}

DIM_FULL_NAMES = {'ie': 'I/E', 'sn': 'S/N', 'tf': 'T/F', 'jp': 'J/P'}

# ── Load Judge Model (separate lightweight SLM) ──
if CONFIG['GUARDRAIL_ENABLED']:
    JUDGE_MODEL_NAME = CONFIG['JUDGE_MODEL']
    print(f'⏳ Loading Judge model: {JUDGE_MODEL_NAME} (4-bit)...')

    judge_bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME, trust_remote_code=True)
    judge_model_raw = AutoModelForCausalLM.from_pretrained(
        JUDGE_MODEL_NAME,
        quantization_config=judge_bnb_config,
        device_map='auto',
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )

    judge_pipe = hf_pipeline(
        'text-generation',
        model=judge_model_raw,
        tokenizer=judge_tokenizer,
        max_new_tokens=CONFIG['JUDGE_MAX_NEW_TOKENS'],
        do_sample=False,
        return_full_text=False,
    )

    _judge_params = sum(p.numel() for p in judge_model_raw.parameters()) / 1e6
    print(f'🛡️ SLM Judge Guardrail: ON (Asymmetric Architecture)')
    print(f'   Generator: {CONFIG["LLM_MODEL"]} (7B, 4-bit) — reasoning + RAG')
    print(f'   Judge:     {JUDGE_MODEL_NAME} ({_judge_params:.0f}M params, 4-bit) — binary YES/NO')
    print(f'   Max retries: {CONFIG["MAX_GUARDRAIL_RETRIES"]}')
else:
    judge_pipe = None
    judge_tokenizer = None
    print('⏭️  Guardrail DISABLED')
    print('   Set CONFIG["GUARDRAIL_ENABLED"] = True to enable')


def _build_judge_prompt(posts_excerpt, dimension_key, predicted_letter):
    """Build a simple YES/NO prompt for the Judge to evaluate one MBTI dimension.

    Key design choices for 3B model reliability:
    - Ask a single YES/NO question (not JSON)
    - Show ONLY the predicted trait description (no opposite — less confusion)
    - Keep the prompt short and focused
    """
    dim_name = DIM_FULL_NAMES[dimension_key]
    desc_pred = JUDGE_DIM_DESCRIPTIONS[dimension_key][predicted_letter]

    return (
        f"Read these social media posts and answer the question.\n\n"
        f"Posts:\n\"{posts_excerpt}\"\n\n"
        f"{predicted_letter} = {desc_pred}\n\n"
        f"Question: Does this person's writing show {predicted_letter} traits?\n"
        f"Answer YES or NO, then give a brief reason."
    )


def _parse_judge_output(raw_output):
    """Parse Judge's YES/NO output. Defaults to ACCEPT if ambiguous.

    Parsing strategy (ordered by confidence):
    1. First word is YES/NO → strong signal
    2. JSON verdict field → structured fallback
    3. YES/NO keyword in first 100 chars → weak signal
    4. Default → ACCEPT (generator already did heavy RAG work)
    """
    text = raw_output.strip()
    if not text:
        return 'ACCEPT', 'Empty judge output'

    # Strip common markdown/formatting prefixes
    clean = re.sub(r'^[\s\n*#\->]+', '', text)

    # ── Strategy 1: First word is YES or NO (strongest signal) ──
    first_chunk = clean[:60].upper()
    yes_start = re.match(r'^\s*\bYES\b', first_chunk)
    no_start = re.match(r'^\s*\bNO\b', first_chunk)

    if yes_start:
        reason = clean[yes_start.end():].strip().lstrip('.,;:!? ')
        return 'ACCEPT', reason[:200] if reason else 'Matches predicted trait'
    if no_start:
        reason = clean[no_start.end():].strip().lstrip('.,;:!? ')
        return 'REJECT', reason[:200] if reason else 'Does not match predicted trait'

    # ── Strategy 2: JSON verdict field (model sometimes outputs JSON anyway) ──
    json_match = re.search(r'"verdict"\s*:\s*"(ACCEPT|REJECT)"', text, re.IGNORECASE)
    if json_match:
        verdict = json_match.group(1).upper()
        reason_match = re.search(r'"reason"\s*:\s*"([^"]*)"', text)
        reason = reason_match.group(1) if reason_match else text[:200]
        return verdict, reason

    # ── Strategy 3: YES/NO keyword in first 100 chars ──
    first100 = clean[:100].upper()
    # Use word boundary to avoid matching NOT, NOTHING, KNOW, etc.
    has_yes = bool(re.search(r'\bYES\b', first100))
    has_no = bool(re.search(r'\bNO\b', first100))

    if has_yes and not has_no:
        return 'ACCEPT', clean[:200]
    if has_no and not has_yes:
        return 'REJECT', clean[:200]

    # ── Strategy 4: Default to ACCEPT ──
    # The generator (7B) already did heavy reasoning with RAG + tools.
    # If the tiny Judge can't give a clear NO, trust the generator.
    return 'ACCEPT', f'Ambiguous judge output: {clean[:150]}'


def guardrail_judge_prediction(posts_text, predicted_dims, detailed=False, dims_to_check=None):
    """Run the SLM Judge on each MBTI dimension independently.

    For each dimension, the Judge receives the posts + predicted letter +
    trait description, and answers YES (ACCEPT) or NO (REJECT) with a reason.

    Returns:
        If detailed=False: dict {dim_key: "ACCEPT"|"REJECT"}
        If detailed=True:  (verdicts_dict, details_dict)
    """
    if not CONFIG['GUARDRAIL_ENABLED'] or judge_pipe is None:
        return ({}, {}) if detailed else {}

    premise = posts_text[:CONFIG['JUDGE_PREMISE_MAX_CHARS']]
    all_dim_keys = ['ie', 'sn', 'tf', 'jp']
    if dims_to_check is None:
        dim_keys = all_dim_keys
    else:
        dim_keys = [dk for dk in dims_to_check if dk in all_dim_keys]

    if not dim_keys:
        return ({}, {}) if detailed else {}

    verdicts = {}
    details = {}

    for dk in dim_keys:
        letter = predicted_dims.get(dk, '?').upper()
        opp_letter = _OPPOSITE.get(letter, '?')

        prompt = _build_judge_prompt(premise, dk, letter)

        # Format as chat message for instruct model
        messages = [{"role": "user", "content": prompt}]
        formatted = judge_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        raw = ''
        try:
            outputs = judge_pipe(formatted)
            raw = outputs[0]['generated_text']
            verdict, reason = _parse_judge_output(raw)
        except Exception as e:
            verdict, reason = 'ACCEPT', f'Judge error: {str(e)[:100]}'

        verdicts[dk] = verdict
        if detailed:
            details[dk] = {
                'predicted': letter,
                'opposite': opp_letter,
                'verdict': verdict,
                'reason': reason,
                'raw_output': raw,
            }

    if detailed:
        return verdicts, details
    return verdicts


# # ── Sanity checks ──
# if CONFIG['GUARDRAIL_ENABLED']:
#     print('\n🔬 Sanity checks:')

#     _test1 = "I love going to parties and meeting new people! We always have fun together as a group!"
#     _v1, _d1 = guardrail_judge_prediction(_test1, {'ie': 'E', 'sn': 'N', 'tf': 'F', 'jp': 'P'}, detailed=True)
#     print(f'\n   Check 1 — extraverted text, predict E (expect ACCEPT on I/E):')
#     for dk in ['ie', 'sn', 'tf', 'jp']:
#         d = _d1[dk]
#         vi = '✅' if d['verdict'] == 'ACCEPT' else '❌'
#         print(f'     {DIM_FULL_NAMES[dk]}: {d["predicted"]} → {vi} {d["verdict"]}  reason: {d["reason"][:100]}')
#     # Show raw output for debugging
#     print(f'\n     [debug] I/E raw: "{_d1["ie"]["raw_output"][:150]}"')

#     _test2 = "I prefer spending time alone reading books. I think deeply about life and rarely go out."
#     _v2, _d2 = guardrail_judge_prediction(_test2, {'ie': 'I', 'sn': 'N', 'tf': 'T', 'jp': 'J'}, detailed=True)
#     print(f'\n   Check 2 — introverted text, predict I (expect ACCEPT on I/E):')
#     for dk in ['ie', 'sn', 'tf', 'jp']:
#         d = _d2[dk]
#         vi = '✅' if d['verdict'] == 'ACCEPT' else '❌'
#         print(f'     {DIM_FULL_NAMES[dk]}: {d["predicted"]} → {vi} {d["verdict"]}  reason: {d["reason"][:100]}')
#     print(f'\n     [debug] I/E raw: "{_d2["ie"]["raw_output"][:150]}"')

#     _v3, _d3 = guardrail_judge_prediction(_test2, {'ie': 'E', 'sn': 'N', 'tf': 'T', 'jp': 'J'}, detailed=True)
#     print(f'\n   Check 3 — introverted text, predict E (WRONG — expect REJECT on I/E):')
#     for dk in ['ie', 'sn', 'tf', 'jp']:
#         d = _d3[dk]
#         vi = '✅' if d['verdict'] == 'ACCEPT' else '❌'
#         print(f'     {DIM_FULL_NAMES[dk]}: {d["predicted"]} → {vi} {d["verdict"]}  reason: {d["reason"][:100]}')
#     print(f'\n     [debug] I/E raw: "{_d3["ie"]["raw_output"][:150]}"')

## 🔧 Cell 8 — LangChain Tools + ReAct Agent

**Hybrid Architecture: Pre-computed Context + Autonomous Agent**

The agent ALWAYS needs text features and similar posts — so we pre-compute them in Python
and inject them directly into the prompt. This saves 2 LLM calls (~20s) per sample.
The agent's iterations are reserved for decisions that actually need LLM reasoning.

**Pre-computed (Python, injected into prompt):**
1. `compute_text_features()` — Psycholinguistic feature extraction (word counts, ratios, etc.)
2. `compute_similar_posts()` — FAISS retrieval → cross-encoder reranking → lost-in-middle reordering

**Agent Tools (LLM decides when/how to use):**
1. `retrieve_mbti_knowledge` — FAISS retriever for domain knowledge (agent specifies dimension query)
2. `recall_experience` — Query long-term memory for similar past classifications
3. `submit_prediction` — Submit final MBTI prediction (**with SLM Judge Guardrail**)

**Why Hybrid (not full pipeline or full agent):**
- **Full pipeline** = no reasoning, fixed steps, can't adapt to ambiguous cases
- **Full agent** = wastes 2/3 iterations on deterministic operations (analyze + retrieve)
- **Hybrid** = deterministic steps run in Python, LLM iterations freed for actual reasoning:
  - Retrieve knowledge when a dimension is uncertain
  - Check memory for similar past cases
  - Revise prediction after Judge rejection

**Guardrail (SLM Judge — Asymmetric Architecture):**
- `submit_prediction` sends each dimension to the **independent Qwen2.5-3B Judge**
- Judge outputs structured JSON: `{"verdict": "ACCEPT"|"REJECT", "reason": "..."}`
- If any dimension is REJECTED → agent receives the reason and must revise
- After `MAX_GUARDRAIL_RETRIES` (2) rejections, force-accept
- Judge is completely separate from the Generator — no self-evaluation bias

In [ ]:

# ═══════════════════════════════════════════════════════════════
#  PRE-COMPUTED CONTEXT FUNCTIONS (called BEFORE agent loop)
# ═══════════════════════════════════════════════════════════════
# These two functions are ALWAYS needed, so we pre-compute them in Python
# and inject the results directly into the prompt. This saves 2 LLM calls
# per sample (~20s) — the agent's iterations are reserved for decisions
# that actually need LLM reasoning (knowledge retrieval, revision, etc.).

def compute_text_features(posts_text: str) -> str:
    """Extract psycholinguistic features — pure Python, no LLM needed."""
    words = posts_text.lower().split()
    total = len(words)
    if total == 0:
        return json.dumps({})
    sentences = max(posts_text.count('.') + posts_text.count('?') + posts_text.count('!'), 1)
    features = {
        'total_words': total,
        'avg_word_len': round(np.mean([len(w) for w in words]), 2),
        'avg_sent_len': round(total / sentences, 1),
        'question_ratio': round(posts_text.count('?') / sentences, 3),
        'exclam_ratio': round(posts_text.count('!') / sentences, 3),
        'first_person_pct': round(
            sum(1 for w in words if w in {'i','me','my','myself','mine'}) / total * 100, 2),
        'we_pct': round(
            sum(1 for w in words if w in {'we','us','our','ourselves'}) / total * 100, 2),
        'emotion_pct': round(
            sum(1 for w in words if w in {
                'feel','feeling','love','hate','happy','sad','angry','beautiful',
                'wonderful','terrible','amazing','awful','care','heart','soul',
                'passion','joy','fear','hope','wish'
            }) / total * 100, 2),
        'thinking_pct': round(
            sum(1 for w in words if w in {
                'think','thought','logic','logical','reason','because','therefore',
                'analyze','analysis','system','theory','hypothesis','evidence','data','fact'
            }) / total * 100, 2),
        'abstract_pct': round(
            sum(1 for w in words if w in {
                'concept','idea','theory','possibility','imagine','philosophy',
                'meaning','purpose','pattern','connection','potential','vision',
                'insight','metaphor','symbol','abstract','infinite'
            }) / total * 100, 2),
        'url_count': posts_text.lower().count('http'),
    }
    return json.dumps(features, indent=2)


def compute_similar_posts(posts_text: str, top_k: int = 5) -> str:
    """Retrieve + rerank similar posts — pure vector search, no LLM needed."""
    retrieve_k = CONFIG['RAG_RETRIEVE_K'] if CONFIG['RERANKER_ENABLED'] else top_k
    docs = post_vectorstore.similarity_search_with_score(posts_text[:1000], k=retrieve_k)
    docs = rerank_docs(posts_text[:500], docs, top_k)
    lines = []
    for i, (doc, score) in enumerate(docs):
        mbti = doc.metadata.get('type', '?')
        preview = doc.page_content[:200]
        lines.append(f"  {i+1}. [{mbti}] (sim={score:.3f}): {preview}")
    return f"Top {len(docs)} similar posts (reranked):\n" + '\n'.join(lines)


# ═══════════════════════════════════════════════════════════════
#  LANGCHAIN TOOLS (via @tool decorator)
#  Only tools that require LLM REASONING to decide when/how to use
# ═══════════════════════════════════════════════════════════════

# ── Shared state: set per-sample before agent.invoke() ──
_current_posts_text = ""
_current_query_embedding = None

# ── Long-term Experience Memory ──
_experience_store = []
_experience_faiss_index = None
_total_memories_stored = 0
_pattern_memory = {}

# ── Guardrail state (reset per sample) ──
_guardrail_retries = 0
_guardrail_pending_dims = []

# ── Early-stop flag: set when submit_prediction is accepted ──
# RobustReActParser checks this flag first — if True, immediately returns
# AgentFinish, preventing the LLM from looping after PREDICTION_SUBMITTED.
_prediction_accepted = False
_prediction_accepted_result = None


@tool
def retrieve_mbti_knowledge(query: str) -> str:
    """Search the MBTI knowledge base for dimension descriptions and type profiles.
    Use a TARGETED query about the dimension(s) you are uncertain about.
    Include the dimension name (I/E, S/N, T/F, J/P) in your query for better results.
    Args:
        query: Search query, e.g. 'T/F thinking vs feeling analytical emotional language'"""
    # ── Dimension-aware filtering ──
    dim_filter = None
    query_upper = query.upper()
    for dim in ['I/E', 'S/N', 'T/F', 'J/P']:
        if dim in query_upper:
            dim_filter = dim
            break
    if dim_filter is None:
        for short, full in [('IE', 'I/E'), ('SN', 'S/N'), ('TF', 'T/F'), ('JP', 'J/P')]:
            if short in query_upper:
                dim_filter = full
                break

    if dim_filter:
        all_docs = knowledge_vectorstore.similarity_search(query, k=12)
        docs = [d for d in all_docs
                if d.metadata.get('dimension') in (dim_filter, 'all')][:4]
        if not docs:
            docs = all_docs[:4]
    else:
        docs = knowledge_vectorstore.similarity_search(query, k=4)

    lines = []
    for doc in docs:
        label = doc.metadata.get('label', '?')
        dim_tag = doc.metadata.get('dimension', '?')
        lines.append(f'  [{label}|{dim_tag}]: {doc.page_content[:300]}')
    filter_note = f' (filtered: {dim_filter})' if dim_filter else ''
    return f"{len(docs)} knowledge chunks{filter_note}:\n" + '\n'.join(lines)


@tool
def recall_experience(dummy: str = "") -> str:
    """Query long-term memory for similar posts previously classified with high confidence.
    Input: any string (ignored, uses current post embedding)."""
    global _experience_faiss_index
    if _experience_faiss_index is None or _experience_faiss_index.ntotal == 0:
        return f"No relevant past experiences (memory_size={len(_experience_store)})"
    query_emb = _current_query_embedding
    k = min(2, _experience_faiss_index.ntotal)
    scores, indices = _experience_faiss_index.search(query_emb.reshape(1, -1), k)
    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx >= 0 and score > 0.5:
            exp = _experience_store[idx]
            results.append(
                f"  [{exp['type']}] (sim={float(score):.2f}, "
                f"conf={exp['avg_conf']:.2f}): {exp['reasoning'][:200]}"
            )
    if not results:
        return f"No similar past experiences found (memory_size={len(_experience_store)})"
    return f"{len(results)} past experiences:\n" + '\n'.join(results)


@tool
def submit_prediction(prediction: str) -> str:
    """Submit MBTI prediction. Format: XXXX ie_conf sn_conf tf_conf jp_conf
    Example: INTP 0.85 0.75 0.90 0.80
    The 4-letter MBTI type, then 4 confidence values (0.5-1.0) for I/E, S/N, T/F, J/P.
    WARNING: An independent Judge model will verify your prediction for each dimension.
    If it finds insufficient evidence, your prediction will be REJECTED with a reason.
    Args:
        prediction: MBTI type and confidences, e.g. 'INTP 0.85 0.75 0.90 0.80'"""
    global _guardrail_retries, _guardrail_pending_dims, _prediction_accepted, _prediction_accepted_result

    # ── Robust parsing ──
    clean = prediction.replace(',', ' ').replace(';', ' ')
    type_match = re.search(r'[IiEe][SsNn][TtFf][JjPp]', clean)
    mbti_type = type_match.group().upper() if type_match else 'INFP'

    numbers = []
    for x in re.findall(r'\b[01]\.\d+\b', clean):
        numbers.append(float(x))

    ie_conf = numbers[0] if len(numbers) > 0 else 0.5
    sn_conf = numbers[1] if len(numbers) > 1 else 0.5
    tf_conf = numbers[2] if len(numbers) > 2 else 0.5
    jp_conf = numbers[3] if len(numbers) > 3 else 0.5

    result = {
        'type': mbti_type,
        'ie': mbti_type[0], 'ie_conf': max(0.0, min(1.0, ie_conf)),
        'sn': mbti_type[1], 'sn_conf': max(0.0, min(1.0, sn_conf)),
        'tf': mbti_type[2], 'tf_conf': max(0.0, min(1.0, tf_conf)),
        'jp': mbti_type[3], 'jp_conf': max(0.0, min(1.0, jp_conf)),
    }

    # ══════════════════════════════════════════════════════════
    #  SLM JUDGE GUARDRAIL (Qwen2.5-3B)
    # ══════════════════════════════════════════════════════════
    if CONFIG['GUARDRAIL_ENABLED'] and _guardrail_retries < CONFIG['MAX_GUARDRAIL_RETRIES']:
        predicted_dims = {
            'ie': result['ie'], 'sn': result['sn'],
            'tf': result['tf'], 'jp': result['jp'],
        }
        dim_conf = {
            'ie': result['ie_conf'],
            'sn': result['sn_conf'],
            'tf': result['tf_conf'],
            'jp': result['jp_conf'],
        }

        # Retry-only-rejected: if previous attempt had rejects, re-check only those dims.
        if _guardrail_pending_dims:
            dims_to_check = [dk for dk in _guardrail_pending_dims if dk in ['ie', 'sn', 'tf', 'jp']]
        elif CONFIG.get('JUDGE_SELECTIVE_ENABLED', False):
            threshold = CONFIG.get('JUDGE_CONF_THRESHOLD', 0.75)
            dims_to_check = [dk for dk in ['ie', 'sn', 'tf', 'jp'] if dim_conf[dk] < threshold]
        else:
            dims_to_check = ['ie', 'sn', 'tf', 'jp']

        if not dims_to_check:
            for dk in ['ie', 'sn', 'tf', 'jp']:
                result[f'{dk}_guard'] = 'ACCEPT'
            _guardrail_pending_dims = []
        else:
            verdicts, judge_details = guardrail_judge_prediction(
                _current_posts_text,
                predicted_dims,
                detailed=True,
                dims_to_check=dims_to_check,
            )

            rejected_dims = []
            rejected_keys = []
            for dk in ['ie', 'sn', 'tf', 'jp']:
                if dk in dims_to_check:
                    v = verdicts.get(dk, 'ACCEPT')
                else:
                    v = 'ACCEPT'
                result[f'{dk}_guard'] = v
                if v == 'REJECT':
                    reason = judge_details.get(dk, {}).get('reason', 'No reason')
                    rejected_keys.append(dk)
                    rejected_dims.append((DIM_FULL_NAMES[dk], reason))

            if rejected_dims:
                _guardrail_retries += 1
                _guardrail_pending_dims = rejected_keys
                reject_str = '; '.join(
                    f'{name}: {reason[:80]}' for name, reason in rejected_dims
                )
                return (
                    f"PREDICTION REJECTED by Judge "
                    f"(attempt {_guardrail_retries}/{CONFIG['MAX_GUARDRAIL_RETRIES']}): "
                    f"[{reject_str}]. "
                    f"Re-analyze the rejected dimension(s) using retrieve_mbti_knowledge, "
                    f"then call submit_prediction with a REVISED prediction."
                )

            _guardrail_pending_dims = []

    # ── Accept prediction (guardrail passed, off, or max retries reached) ──
    _guardrail_pending_dims = []
    accepted_str = "PREDICTION_SUBMITTED:" + json.dumps(result)
    # Signal the parser to stop the agent loop immediately
    _prediction_accepted = True
    _prediction_accepted_result = accepted_str
    return accepted_str


# ═══════════════════════════════════════════════════════════════
#  ROBUST OUTPUT PARSER (fixes LLM hallucination of Final Answer + Action)
# ═══════════════════════════════════════════════════════════════
from langchain.agents.output_parsers import ReActSingleInputOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.agents import AgentFinish

class RobustReActParser(ReActSingleInputOutputParser):
    """Custom parser that handles common 7B-model failure modes:
    1. LLM generates BOTH 'Action:' and 'Final Answer:' in one response
    2. LLM generates only 'Thought:' without 'Action:' (token limit hit)
    3. LLM hallucinates 'Observation:' blocks
    4. LLM repeats 'Final Answer:' lines
    5. 'Action:' present but 'Action Input:' missing (LLM continues reasoning after Action)
    6. LLM hallucinates PREDICTION_SUBMITTED / extra Thought-Action blocks after Action Input
    7. Agent loops after PREDICTION_SUBMITTED — early-stop via _prediction_accepted flag
    """

    # Default inputs for tools when Action Input is missing
    _DEFAULT_TOOL_INPUTS = {
        'retrieve_mbti_knowledge': 'MBTI personality dimensions I/E S/N T/F J/P',
        'recall_experience': '',
        'submit_prediction': 'INFP 0.70 0.70 0.70 0.70',
    }

    def _extract_action_input_from_context(self, tool_name: str, text: str) -> str:
        """Try to infer a useful Action Input from surrounding text when LLM forgot it."""
        # For retrieve_mbti_knowledge, look for dimension mentions in the text
        if tool_name == 'retrieve_mbti_knowledge':
            dims_found = []
            for dim in ['I/E', 'S/N', 'T/F', 'J/P']:
                if dim in text.upper() or dim.replace('/', '') in text.upper():
                    dims_found.append(dim)
            if dims_found:
                return f"{' '.join(dims_found)} personality traits indicators"
        # For submit_prediction, look for an MBTI type in the text
        if tool_name == 'submit_prediction':
            mbti_match = re.search(r'\b([IiEe][SsNn][TtFf][JjPp])\b', text)
            if mbti_match:
                return f"{mbti_match.group(1).upper()} 0.70 0.70 0.70 0.70"
        return self._DEFAULT_TOOL_INPUTS.get(tool_name, '')

    def _sanitize_submit_prediction_input(self, text: str) -> str:
        """Extract only MBTI type + 4 confidences from noisy Action Input text."""
        type_match = re.search(r'\b([IiEe][SsNn][TtFf][JjPp])\b', text)
        mbti_type = type_match.group(1).upper() if type_match else 'INFP'
        numbers = [float(x) for x in re.findall(r'\b[01]\.\d+\b', text)]
        while len(numbers) < 4:
            numbers.append(0.70)
        numbers = numbers[:4]
        return f"{mbti_type} {numbers[0]:.2f} {numbers[1]:.2f} {numbers[2]:.2f} {numbers[3]:.2f}"

    def _trim_to_first_action_block(self, text: str) -> str:
        """Keep only the first Thought/Action/Action Input block and drop hallucinated tails."""
        action_match = re.search(r'Action:\s*([^\n]+)', text)
        action_input_match = re.search(r'Action Input:\s*(.*)', text, re.DOTALL)
        if not action_match or not action_input_match:
            return text

        tool_name = action_match.group(1).strip()
        action_input = action_input_match.group(1).strip()

        stop_patterns = [
            r'\nThought:',
            r'\nAction:',
            r'\nFinal Answer:',
            r'PREDICTION_SUBMITTED:',
            r'\nObservation:',
        ]
        stop_positions = []
        for pattern in stop_patterns:
            match = re.search(pattern, action_input)
            if match:
                stop_positions.append(match.start())
        if stop_positions:
            action_input = action_input[:min(stop_positions)].strip()

        if tool_name == 'submit_prediction':
            action_input = self._sanitize_submit_prediction_input(action_input)

        thought_prefix = text.split('Action:')[0].rstrip()
        return f"{thought_prefix}\nAction: {tool_name}\nAction Input: {action_input}".strip()

    def parse(self, text: str):
        # ── Failure mode 7: Agent looping after accepted prediction ──
        # submit_prediction sets _prediction_accepted=True when it returns PREDICTION_SUBMITTED.
        # Any subsequent LLM call reaches here first — we immediately return AgentFinish
        # without even looking at what the model generated, stopping the loop cold.
        global _prediction_accepted, _prediction_accepted_result
        if _prediction_accepted and _prediction_accepted_result:
            return AgentFinish(
                return_values={"output": _prediction_accepted_result},
                log=text,
            )

        if "PREDICTION_SUBMITTED:" in text and "Final Answer:" not in text:
            submitted_idx = text.index("PREDICTION_SUBMITTED:")
            submitted_payload = text[submitted_idx:].strip()
            stop_positions = []
            for marker in ["\nThought:", "\nAction:", "\nObservation:", "\nFinal Answer:"]:
                marker_idx = submitted_payload.find(marker)
                if marker_idx != -1:
                    stop_positions.append(marker_idx)
            if stop_positions:
                submitted_payload = submitted_payload[:min(stop_positions)].strip()
            text = f"Final Answer: {submitted_payload}"

        if "\nObservation:" in text:
            text = text.split("\nObservation:")[0]
        if "\nObservation :" in text:
            text = text.split("\nObservation :")[0]

        # Fix "Action Input" without colon (common 7B-model formatting issue)
        text = re.sub(r'Action Input\s*(?!:)', 'Action Input: ', text)

        has_action = "Action:" in text and "Action Input:" in text
        has_action_no_input = "Action:" in text and "Action Input:" not in text
        has_final  = "Final Answer:" in text

        # ── Fix missing Action Input (failure mode 5) ──
        # LLM wrote "Action: tool_name" but continued reasoning without "Action Input:"
        if has_action_no_input and not has_final:
            action_idx = text.index("Action:")
            action_line = text[action_idx:].split('\n')[0]
            tool_name = action_line.replace("Action:", "").strip()
            # Extract a useful default input from context
            inferred_input = self._extract_action_input_from_context(tool_name, text)
            # Rebuild: keep everything up to and including the Action line, add Action Input
            before_action = text[:action_idx]
            text = (f"{before_action}Action: {tool_name}\n"
                    f"Action Input: {inferred_input}")
            has_action = True
            has_action_no_input = False

        if has_action and not has_final:
            text = self._trim_to_first_action_block(text)

        if has_action and has_final:
            action_idx = text.index("Action:")
            final_idx  = text.index("Final Answer:")
            if action_idx < final_idx:
                text = text[:final_idx].rstrip()
            else:
                text = text[:action_idx].rstrip()

        if has_final and not has_action:
            fa_idx = text.index("Final Answer:")
            answer = text[fa_idx + len("Final Answer:"):].strip()
            if "\nFinal Answer:" in answer:
                answer = answer.split("\nFinal Answer:")[0].strip()
            text = text[:fa_idx] + "Final Answer: " + answer

        if not has_action and not has_final:
            # Thought-only output (token limit hit before Action:)
            # Strategy: extract whatever MBTI signal we can from the reasoning
            mbti_match = re.search(r'\b([IiEe][SsNn][TtFf][JjPp])\b', text)
            if mbti_match:
                mbti_type = mbti_match.group(1).upper()
            else:
                # No full type found — infer individual dimensions from reasoning
                # Look for patterns like "introversion (I)", "thinking (T)", etc.
                # All regexes use re.IGNORECASE to handle mixed-case LLM output
                dim_patterns = {
                    'ie': [
                        (r'(?:introvert|introversion)\s*\(?i\)?', 'I'),
                        (r'(?:extravert|extraversion|extrovert|extroversion)\s*\(?e\)?', 'E'),
                        (r'(?:indicate|suggest|could indicate|lean|point)s?\s+(?:towards?\s+)?(?:an?\s+)?introversion', 'I'),
                        (r'(?:indicate|suggest|could indicate|lean|point)s?\s+(?:towards?\s+)?(?:an?\s+)?extraversion', 'E'),
                        (r'introvert|introspective|reflective.*solitude', 'I'),
                        (r'extravert|sociable|energetic.*social', 'E'),
                    ],
                    'sn': [
                        (r'(?:sensing)\s*\(?s\)?', 'S'),
                        (r'(?:intuition|intuitive)\s*\(?n\)?', 'N'),
                        (r'abstract\s+(?:concept|idea|language|thinking)', 'N'),
                        (r'concrete\s+(?:detail|fact|language|thinking)', 'S'),
                        (r'(?:indicate|suggest|lean)s?\s+(?:towards?\s+)?(?:an?\s+)?(?:sensing|s\b)', 'S'),
                        (r'(?:indicate|suggest|lean)s?\s+(?:towards?\s+)?(?:an?\s+)?(?:intuition|n\b)', 'N'),
                    ],
                    'tf': [
                        (r'(?:thinking)\s*\(?t\)?', 'T'),
                        (r'(?:feeling)\s*\(?f\)?', 'F'),
                        (r'logical\s+(?:reasoning|analysis|approach)', 'T'),
                        (r'emotional\s+(?:language|expression|focus|response)', 'F'),
                        (r'(?:indicate|suggest|align|lean)s?\s+(?:with\s+)?(?:towards?\s+)?thinking', 'T'),
                        (r'(?:indicate|suggest|align|lean)s?\s+(?:with\s+)?(?:towards?\s+)?feeling', 'F'),
                    ],
                    'jp': [
                        (r'(?:judging)\s*\(?j\)?', 'J'),
                        (r'(?:perceiving)\s*\(?p\)?', 'P'),
                        (r'(?:indicate|suggest|align|lean)s?\s+(?:towards?\s+)?(?:a\s+)?(?:judging|j\b)', 'J'),
                        (r'(?:indicate|suggest|align|lean)s?\s+(?:towards?\s+)?(?:a\s+)?(?:perceiving|p\b)', 'P'),
                        (r'open-ended|flexible|spontaneous', 'P'),
                        (r'decisive|structured|methodical|organized', 'J'),
                    ],
                }
                inferred = {}
                for dim, patterns in dim_patterns.items():
                    for pattern, letter in patterns:
                        if re.search(pattern, text, re.IGNORECASE):
                            inferred[dim] = letter
                            break
                # Build type from inferred dimensions (default to I/N/F/P for unknowns)
                mbti_type = (inferred.get('ie', 'I') + inferred.get('sn', 'N')
                             + inferred.get('tf', 'F') + inferred.get('jp', 'P'))

            conf = 0.65 if mbti_match else 0.55
            text = (f"Thought: Auto-extracted prediction from truncated thought.\n"
                    f"Action: submit_prediction\n"
                    f"Action Input: {mbti_type} {conf} {conf} {conf} {conf}")

        return super().parse(text)


# ═══════════════════════════════════════════════════════════════
#  LANGCHAIN ReAct AGENT
# ═══════════════════════════════════════════════════════════════

# Only tools that require LLM reasoning about WHEN/HOW to use them.
# analyze_text and retrieve_similar_posts are pre-computed and injected
# into the prompt — saves 2 LLM calls (~20s) per sample.
tools = [retrieve_mbti_knowledge, recall_experience, submit_prediction]

# Build prompt — include guardrail warning only when enabled
_guardrail_prompt_section = ""
if CONFIG['GUARDRAIL_ENABLED']:
    _guardrail_prompt_section = """
IMPORTANT — SLM Judge Guardrail:
An independent Judge model will verify each dimension of your prediction.
If it finds weak evidence for any dimension, your prediction will be REJECTED with a reason.
When rejected, use retrieve_mbti_knowledge to investigate, then resubmit."""

# ── Hybrid Prompt: Pre-computed context + Autonomous Agent ──
# Text features and similar posts are ALWAYS needed → pre-computed in Python.
# The agent uses its iterations for decisions that need LLM reasoning:
#   - Whether to retrieve MBTI knowledge (and which dimension to query)
#   - Whether to recall past experiences
#   - How to revise after Judge rejection
REACT_PROMPT = PromptTemplate.from_template(
"""You are an MBTI personality classifier. Analyze the pre-computed evidence below, then predict the MBTI type.

MBTI has 4 binary dimensions:
- I/E: Introversion (reflective, "I think", solitude) vs Extraversion (social, "we/us", energetic)
- S/N: Sensing (concrete, facts, details) vs Intuition (abstract, metaphors, "what if")
- T/F: Thinking (logical, "because/therefore") vs Feeling (emotional, "feel/love/care")
- J/P: Judging (decisive, "should/must", plans) vs Perceiving (open-ended, "maybe", flexible)

═══ PRE-COMPUTED EVIDENCE (already gathered) ═══

Text Features:
{text_features}

Similar Labeled Posts (few-shot examples):
{similar_posts}

═══ TOOLS (use only if needed) ═══

Tools available:
{tools}

Tool names: {tool_names}

FORMAT RULES (follow EXACTLY):
- Write Thought, then Action, then Action Input. STOP there. Do NOT write anything else after Action Input.
- Do NOT write "Observation:" — the system provides observations automatically.
- Do NOT write "Final Answer:" until you have called submit_prediction and seen PREDICTION_SUBMITTED in the observation.
- Keep Thought BRIEF (1-3 sentences). Evidence is already analyzed above — just state your decision.
- Write Thought, then Action, then Action Input on separate lines. STOP after Action Input.
- Do NOT write "Observation:" — the system provides observations automatically.
- Do NOT write "Final Answer:" until you have called submit_prediction and seen PREDICTION_SUBMITTED in the observation.

Format for tool calls:
Thought: Brief reasoning (1-3 sentences)
Action: [tool name]
Action Input: [input]

Format for final answer (ONLY after seeing PREDICTION_SUBMITTED):
Final Answer: [the PREDICTION_SUBMITTED result]
""" + _guardrail_prompt_section + """
STRATEGY:
1. Read the pre-computed evidence above carefully.
2. If any dimension is unclear, use retrieve_mbti_knowledge with a TARGETED query (include the dimension: I/E, S/N, T/F, J/P).
3. Optionally use recall_experience to check past similar cases.
4. Call submit_prediction with: XXXX ie_conf sn_conf tf_conf jp_conf (e.g. INTP 0.85 0.75 0.90 0.80)
5. If rejected by Judge, use the reason to investigate the weak dimension(s) further, then resubmit.

Posts to classify:
{input}

{agent_scratchpad}""")

react_agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=REACT_PROMPT,
    output_parser=RobustReActParser(),
)

print('✅ LangChain Agent created')
print(f'   Agent type:     create_react_agent (ReAct) + RobustReActParser')
print(f'   LLM:            HuggingFacePipeline ({LLM_MODEL_NAME})')
print(f'   Pre-computed:   analyze_text + retrieve_similar_posts (injected into prompt)')
print(f'   Agent tools:    {[t.name for t in tools]} (only reasoning-dependent tools)')
print(f'   Max iterations: {CONFIG["MAX_AGENT_ITERATIONS"]}')
print(f'   Tool selection: AUTONOMOUS — agent decides if/when to use knowledge or memory')
if CONFIG['GUARDRAIL_ENABLED']:
    print(f'   Guardrail:      SLM Judge ({CONFIG["JUDGE_MODEL"]}) — '
          f'ACCEPT/REJECT per dimension '
          f'(max {CONFIG["MAX_GUARDRAIL_RETRIES"]} retries)')
else:
    print(f'   Guardrail:      OFF — predictions accepted without verification')


## 🔬 Cell 8b — DEMO: Single User Predictions with Full Agent Trace

Run the LangChain agent on a small sample to visualize the autonomous ReAct loop — where the LLM reasons, selects tools, and generates dynamic arguments via `AgentExecutor`.

**SLM Judge Guardrail**: An independent Qwen2.5-3B model reviews each dimension and returns `ACCEPT`/`REJECT` with a natural language reason. When rejected, the agent receives actionable feedback to revise specific dimensions.

In [ ]:

# ═══════════════════════════════════════════════════════════════
#  HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════

DIM_LETTER_MAP_INV = {
    'ie': {1: 'I', 0: 'E'},
    'sn': {1: 'N', 0: 'S'},
    'tf': {1: 'F', 0: 'T'},
    'jp': {1: 'J', 0: 'P'},
}


def store_experience(query_embedding, posts_text, result, trace_text=""):
    global _experience_faiss_index, _total_memories_stored
    confs = [result.get(f'{d}_conf', 0) for d in ['ie', 'sn', 'tf', 'jp']]
    min_conf = min(confs)
    avg_conf = float(np.mean(confs))
    if min_conf < CONFIG['MEMORY_CONF_THRESHOLD']:
        return False
    experience = {
        'type': result['type'],
        'posts_preview': posts_text[:300],
        'result': result,
        'reasoning': trace_text[:500] if trace_text else 'Direct classification',
        'avg_conf': avg_conf,
    }
    _experience_store.append(experience)
    emb = query_embedding.copy().reshape(1, -1).astype(np.float32)
    if _experience_faiss_index is None:
        _experience_faiss_index = faiss.IndexFlatIP(emb.shape[1])
    _experience_faiss_index.add(emb)
    _total_memories_stored += 1
    return True


def parse_agent_result(output_text):
    m = re.search(r'PREDICTION_SUBMITTED:\s*(\{[^}]+\})', output_text)
    if m:
        try:
            return json.loads(m.group(1))
        except (json.JSONDecodeError, ValueError):
            pass
    for match in re.finditer(r'\{[^{}]+\}', output_text):
        try:
            obj = json.loads(match.group())
            if 'type' in obj or 'ie' in obj:
                return validate_result(obj)
        except (json.JSONDecodeError, ValueError):
            continue
    return None


def validate_result(obj):
    t = obj.get('type', obj.get('mbti_type', '')).upper()
    for dim, idx, pair in [('ie', 0, 'IE'), ('sn', 1, 'SN'),
                            ('tf', 2, 'TF'), ('jp', 3, 'JP')]:
        letter = obj.get(dim, '').upper()
        if letter not in pair:
            letter = (t[idx].upper()
                      if len(t) > idx and t[idx].upper() in pair
                      else pair[0])
        obj[dim] = letter
        conf = obj.get(f'{dim}_conf', 0.5)
        try:
            conf = max(0.0, min(1.0, float(conf)))
        except (ValueError, TypeError):
            conf = 0.5
        obj[f'{dim}_conf'] = conf
    obj['type'] = (obj['ie'] + obj['sn'] + obj['tf'] + obj['jp']).upper()
    return obj


def run_agent_on_sample(posts_text, query_embedding, verbose=False):
    """Run LangChain AgentExecutor on a single sample.
    Pre-computes text features and similar posts, then injects them into the prompt.
    Agent iterations are reserved for reasoning-dependent tools only."""
    global _current_posts_text, _current_query_embedding, _guardrail_retries, _guardrail_pending_dims
    global _prediction_accepted, _prediction_accepted_result
    _current_posts_text = posts_text
    _current_query_embedding = query_embedding
    _guardrail_retries = 0
    _guardrail_pending_dims = []
    # Reset early-stop flag for this sample
    _prediction_accepted = False
    _prediction_accepted_result = None

    # ── Pre-compute deterministic context (saves ~2 LLM calls) ──
    text_features = compute_text_features(posts_text)
    similar_posts = compute_similar_posts(posts_text, top_k=CONFIG['RAG_TOP_K'])

    executor = AgentExecutor(
        agent=react_agent,
        tools=tools,
        max_iterations=CONFIG['MAX_AGENT_ITERATIONS'],
        verbose=verbose,
        handle_parsing_errors=True,
        return_intermediate_steps=True,
    )

    try:
        result = executor.invoke({
            'input': posts_text[:1200],
            'text_features': text_features,
            'similar_posts': similar_posts,
        })
        output = result.get('output', '')
        steps = result.get('intermediate_steps', [])

        parsed = parse_agent_result(output)
        if parsed is None:
            for action, observation in steps:
                if hasattr(action, 'tool') and action.tool == 'submit_prediction':
                    parsed = parse_agent_result(str(observation))
                    if parsed:
                        break
        if parsed:
            parsed = validate_result(parsed)
        return parsed, output, steps
    except Exception as e:
        return None, f"Error: {str(e)}", []


def fallback_predict(posts_text, query_embedding):
    """Direct LLM prediction when agent loop fails — still uses RAG context with reranking."""
    retrieve_k = CONFIG['RAG_RETRIEVE_K'] if CONFIG['RERANKER_ENABLED'] else 5
    similar_docs = post_vectorstore.similarity_search_with_score(posts_text[:1000], k=retrieve_k)
    similar_docs = rerank_docs(posts_text[:500], similar_docs, top_k=5)
    knowledge_docs = knowledge_vectorstore.similarity_search(
        "MBTI personality dimensions introversion extraversion", k=4
    )
    sim_str = '\n'.join(
        f"  [{d.metadata.get('type','?')}] (sim={s:.2f}): {d.page_content[:200]}"
        for d, s in similar_docs
    )
    know_str = '\n'.join(
        f"  [{d.metadata.get('label','?')}]: {d.page_content[:200]}"
        for d in knowledge_docs
    )
    prompt = (
        'You are an MBTI classifier. Respond with ONLY JSON:\n'
        '{"type":"XXXX","ie":"I or E","ie_conf":0.0-1.0,'
        '"sn":"S or N","sn_conf":0.0-1.0,'
        '"tf":"T or F","tf_conf":0.0-1.0,'
        '"jp":"J or P","jp_conf":0.0-1.0}\n\n'
        f'Similar posts:\n{sim_str}\n\n'
        f'Knowledge:\n{know_str}\n\n'
        f'Posts:\n{posts_text[:1000]}\n\n'
        'Classify. Output ONLY JSON:'
    )
    raw = llm.invoke(prompt)
    parsed = parse_agent_result(str(raw))
    if parsed:
        parsed = validate_result(parsed)
        if CONFIG['GUARDRAIL_ENABLED']:
            predicted_dims = {k: parsed[k] for k in ['ie', 'sn', 'tf', 'jp']}
            verdicts = guardrail_judge_prediction(posts_text, predicted_dims)
            for k in ['ie', 'sn', 'tf', 'jp']:
                parsed[f'{k}_guard'] = verdicts.get(k, 'ACCEPT')
        return parsed
    return {
        'type': 'INFP', 'ie': 'I', 'ie_conf': 0.5,
        'sn': 'N', 'sn_conf': 0.5, 'tf': 'F', 'tf_conf': 0.5,
        'jp': 'P', 'jp_conf': 0.5,
    }


# ═══════════════════════════════════════════════════════════════
#  DEMO: LangChain Agent Trace
# ═══════════════════════════════════════════════════════════════
import random
DEMO_INDICES = [random.randint(0,100) for _ in range(10)]
NUM_DEMOS = len(DEMO_INDICES)

# Reset memory
_pattern_memory = {}
_experience_store = []
_experience_faiss_index = None
_total_memories_stored = 0

print('=' * 70)
print(f'🔬 DEMO: {NUM_DEMOS} Users — LangChain ReAct Agent Trace')
print('=' * 70)
print(f'   Agent: LangChain AgentExecutor + create_react_agent')
print(f'   Tools: {[t.name for t in tools]}')
if CONFIG['GUARDRAIL_ENABLED']:
    print(f'   Guardrail: SLM Judge ({CONFIG["JUDGE_MODEL"]}) — '
          f'ACCEPT/REJECT per dimension '
          f'(max {CONFIG["MAX_GUARDRAIL_RETRIES"]} retries)')
else:
    print(f'   Guardrail: OFF')
print(f'   Long-term memory starts EMPTY — watch it grow!\n')

for demo_num, sample_idx in enumerate(DEMO_INDICES, 1):
    if sample_idx >= len(test_df):
        print(f'\n⚠️  Sample index {sample_idx} out of range, skipping.')
        continue

    row = test_df.iloc[sample_idx]
    posts_text = row['concat_posts']
    query_emb  = test_embeddings_np[sample_idx]

    true_type = row[LABEL_COL].upper()
    true_dims = {
        'ie': DIM_LETTER_MAP_INV['ie'][row['label_ie']],
        'sn': DIM_LETTER_MAP_INV['sn'][row['label_sn']],
        'tf': DIM_LETTER_MAP_INV['tf'][row['label_tf']],
        'jp': DIM_LETTER_MAP_INV['jp'][row['label_jp']],
    }

    t0 = time.time()
    parsed, output, steps = run_agent_on_sample(posts_text, query_emb, verbose=True)
    elapsed = time.time() - t0

    if parsed is None:
        parsed = fallback_predict(posts_text, query_emb)

    pred_type = parsed.get('type', '????')

    print(f'\n{"=" * 70}')
    print(f'  🧑 DEMO {demo_num}/{NUM_DEMOS}  |  Test Sample #{sample_idx}')
    print(f'{"=" * 70}')

    preview = posts_text[:300].replace('\n', ' ')
    print(f'\n📝 Posts preview:\n   "{preview}..."')

    # ── Show pre-computed context summary ──
    pre_features = compute_text_features(posts_text)
    print(f'\n📊 Pre-computed text features: {pre_features[:200]}')
    print(f'📊 Pre-computed similar posts: (injected into prompt, {CONFIG["RAG_TOP_K"]} reranked)')

    match_icon = '✅' if pred_type == true_type else '❌'
    print(f'\n📋 True MBTI:      {true_type}')
    print(f'🤖 Predicted MBTI: {pred_type}  {match_icon}')
    print(f'⏱  Inference time: {elapsed:.1f}s')

    # ── Show guardrail activity ──
    if CONFIG['GUARDRAIL_ENABLED']:
        guardrail_rejections = sum(
            1 for a, obs in steps
            if hasattr(a, 'tool') and a.tool == 'submit_prediction'
            and 'PREDICTION REJECTED' in str(obs)
        )
        if guardrail_rejections > 0:
            print(f'🛡️  Judge rejections: {guardrail_rejections} '
                  f'(Judge found weak evidence — agent revised)')

    print(f'\n📊 Axis Predictions:')
    dim_full = {'ie': 'I/E', 'sn': 'S/N', 'tf': 'T/F', 'jp': 'J/P'}
    for key in ['ie', 'sn', 'tf', 'jp']:
        pred_letter = parsed.get(key, '?')
        conf = parsed.get(f'{key}_conf', 0.0)
        guard_verdict = parsed.get(f'{key}_guard', None)
        true_letter = true_dims[key]
        axis_match = '✅' if pred_letter == true_letter else '❌'
        conf_bar = '█' * int(conf * 20) + '░' * (20 - int(conf * 20))
        guard_str = ''
        if guard_verdict is not None:
            guard_icon = '✅' if guard_verdict == 'ACCEPT' else '❌'
            guard_str = f' Judge:{guard_icon}{guard_verdict}'
        print(f'   {dim_full[key]}: {pred_letter} (conf={conf:.2f}{guard_str}) '
              f'[{conf_bar}] true={true_letter} {axis_match}')

    # ── SLM Judge Post-Verification Analysis ──
    if CONFIG['GUARDRAIL_ENABLED']:
        predicted_dims = {k: parsed.get(k, '?') for k in ['ie', 'sn', 'tf', 'jp']}
        _, judge_details = guardrail_judge_prediction(posts_text, predicted_dims, detailed=True)

        print(f'\n🛡️ SLM Judge Analysis (Independent Verifier):')
        print(f'   Judge model: {CONFIG["JUDGE_MODEL"]}')
        print(f'   ┌───────┬──────┬──────────┬─────────────────────────────────────────┐')
        print(f'   │  Dim  │ Pred │ Verdict  │ Reason                                  │')
        print(f'   ├───────┼──────┼──────────┼─────────────────────────────────────────┤')
        for key in ['ie', 'sn', 'tf', 'jp']:
            d = judge_details.get(key, {})
            pred_l = d.get('predicted', '?')
            verdict = d.get('verdict', '?')
            reason = d.get('reason', '')[:40]
            vi = '✅' if verdict == 'ACCEPT' else '❌'
            print(f'   │ {dim_full[key]:>5} │  {pred_l}   │ {vi} {verdict:<6} │ {reason:<39} │')
        print(f'   └───────┴──────┴──────────┴─────────────────────────────────────────┘')

        rejected_dims = [dim_full[k] for k in ['ie', 'sn', 'tf', 'jp']
                         if judge_details.get(k, {}).get('verdict') == 'REJECT']
        if rejected_dims:
            print(f'   ⛔ Rejected: {", ".join(rejected_dims)}')
        else:
            print(f'   ✅ All dimensions accepted by Judge')

        print(f'\n🔎 Judge Reasoning Details:')
        for key in ['ie', 'sn', 'tf', 'jp']:
            d = judge_details.get(key, {})
            pred_l = d.get('predicted', '?')
            verdict = d.get('verdict', '?')
            reason = d.get('reason', 'N/A')
            vi = '✅' if verdict == 'ACCEPT' else '❌'
            print(f'\n   {dim_full[key]} [predicted {pred_l}] → {vi} {verdict}')
            print(f'     reason: "{reason}"')

    # ── Agent trace ──
    print(f'\n🔄 LangChain Agent Trace ({len(steps)} tool calls):')
    for i, (action, observation) in enumerate(steps):
        tool_name = action.tool if hasattr(action, 'tool') else str(action)
        tool_input = action.tool_input if hasattr(action, 'tool_input') else ''
        log = action.log if hasattr(action, 'log') else ''
        icons = {
            'retrieve_mbti_knowledge': '📚', 'recall_experience': '💡',
            'submit_prediction': '🧠',
        }
        icon = icons.get(tool_name, '▶')

        is_rejected = (tool_name == 'submit_prediction'
                       and 'PREDICTION REJECTED' in str(observation))
        reject_tag = ' 🛡️ REJECTED' if is_rejected else ''

        print(f'\n   Step {i+1}: {icon} [{tool_name}]{reject_tag}')
        thought_match = re.search(r'Thought:\s*(.+?)(?=\nAction)', log, re.DOTALL)
        if thought_match:
            print(f'   💭 Thought: {thought_match.group(1).strip()[:400]}')
        if tool_input:
            print(f'   🔧 Input: {str(tool_input)[:200]}')
        print(f'   👁  Observation: {str(observation)[:300]}')

    trace_text = ' → '.join(
        f"{a.tool}({str(a.tool_input)[:50]})"
        for a, _ in steps if hasattr(a, 'tool')
    )
    stored = store_experience(query_emb, posts_text, parsed, trace_text)
    _pattern_memory[pred_type] = _pattern_memory.get(pred_type, 0) + 1
    mem_icon = '✅' if stored else '⏭'
    print(f'\n   💾 Memory: pattern["{pred_type}"]={_pattern_memory[pred_type]}, '
          f'{mem_icon} experience store size={len(_experience_store)}')
    print(f'{"─" * 70}')

print(f'\n{"=" * 70}')
print(f'📊 Long-term Memory Summary after {NUM_DEMOS} Demos:')
print(f'   Experiences stored:  {len(_experience_store)}/{NUM_DEMOS}')
print(f'   Type distribution:   '
      f'{dict(sorted(_pattern_memory.items(), key=lambda x: -x[1]))}')
print(f'{"=" * 70}')

# Reset for full run
_pattern_memory = {}
_experience_store = []
_experience_faiss_index = None
_total_memories_stored = 0
print(f'\n🔬 Demo complete. Memory reset for full test run.')


## 🚀 Cell 9 — Run Agent Inference on Test Set

Process all test samples through the LangChain `AgentExecutor`. Each sample runs the full ReAct loop autonomously.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  RUN LANGCHAIN AGENT INFERENCE ON TEST SET
# ═══════════════════════════════════════════════════════════════
n_test = len(test_df)

print(f'⏳ [RAG+Agent] LangChain AgentExecutor on {n_test} test samples...')
print(f'   Agent:           create_react_agent (ReAct)')
print(f'   Max iterations:  {CONFIG["MAX_AGENT_ITERATIONS"]}')
print(f'   Tools:           {[t.name for t in tools]}')
if CONFIG['GUARDRAIL_ENABLED']:
    print(f'   Guardrail:       SLM Judge ({CONFIG["JUDGE_MODEL"]}) — '
          f'ACCEPT/REJECT per dimension')
else:
    print(f'   Guardrail:       OFF')
print(f'   VectorStores:    post_vectorstore + knowledge_vectorstore\n')

agent_results = []
errors = 0
agent_fallbacks = 0
guardrail_total_rejections = 0
t_start = time.time()

for i in tqdm(range(n_test), desc='Agent inference'):
    posts_text = test_df.iloc[i]['concat_posts']
    query_emb = test_embeddings_np[i]

    try:
        parsed, output, steps = run_agent_on_sample(posts_text, query_emb, verbose=False)

        if CONFIG['GUARDRAIL_ENABLED']:
            for a, obs in steps:
                if hasattr(a, 'tool') and a.tool == 'submit_prediction':
                    if 'PREDICTION REJECTED' in str(obs):
                        guardrail_total_rejections += 1

        if parsed is None:
            parsed = fallback_predict(posts_text, query_emb)
            agent_fallbacks += 1

        agent_results.append(parsed)

        mbti = parsed.get('type', 'INFP')
        _pattern_memory[mbti] = _pattern_memory.get(mbti, 0) + 1
        trace_text = ' → '.join(
            f"{a.tool}" for a, _ in steps if hasattr(a, 'tool')
        )
        store_experience(query_emb, posts_text, parsed, trace_text)

    except Exception as e:
        errors += 1
        agent_results.append({
            'type': 'INFP', 'ie': 'I', 'ie_conf': 0.5,
            'sn': 'N', 'sn_conf': 0.5, 'tf': 'F', 'tf_conf': 0.5,
            'jp': 'P', 'jp_conf': 0.5,
        })
        if errors <= 5:
            print(f'  ⚠️ Sample {i} error: {str(e)[:200]}')

    if (i + 1) % 50 == 0:
        elapsed = time.time() - t_start
        done = len(agent_results)
        speed = done / elapsed
        eta = (n_test - done) / speed if speed > 0 else 0
        top_types = sorted(_pattern_memory.items(), key=lambda x: -x[1])[:5]
        print(f'  [{done}/{n_test}] '
              f'speed={speed:.2f} samples/s | ETA={eta/60:.1f}min | '
              f'errors={errors} | fallbacks={agent_fallbacks}'
              + (f' | judge_rejections={guardrail_total_rejections}' if CONFIG['GUARDRAIL_ENABLED'] else ''))

elapsed_total = time.time() - t_start
print(f'\n✅ Inference complete in {elapsed_total/60:.1f} minutes')
print(f'   Speed: {n_test/elapsed_total:.2f} samples/s')
print(f'   Errors: {errors}/{n_test}')
print(f'   Fallbacks: {agent_fallbacks}')
if CONFIG['GUARDRAIL_ENABLED']:
    print(f'   Judge rejections: {guardrail_total_rejections} '
          f'(dimensions rejected — agent revised)')
else:
    print(f'   Guardrail: OFF (no verification)')
print(f'   Experience store: {len(_experience_store)} '
      f'({len(_experience_store)/max(n_test,1)*100:.1f}%)')
print(f'   Type distribution:')
for t, cnt in sorted(_pattern_memory.items(), key=lambda x: -x[1]):
    print(f'     {t}: {cnt} ({cnt/n_test*100:.1f}%)')

## 📊 Cell 10 — Metrics (F1, Accuracy, AUC) + Save CSV

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  METRICS: F1, Accuracy, AUC  —  4 axes + full 16 types
# ═══════════════════════════════════════════════════════════════

DIM_LABEL_MAP = {
    'ie': {'I': 1, 'E': 0},
    'sn': {'N': 1, 'S': 0},
    'tf': {'F': 1, 'T': 0},
    'jp': {'J': 1, 'P': 0},
}
DIM_KEYS = ['ie', 'sn', 'tf', 'jp']

y_true_axes = test_df[LABEL_COLS].values
y_pred_axes = np.zeros((len(agent_results), 4), dtype=int)
y_prob_axes = np.zeros((len(agent_results), 4), dtype=float)

for i, res in enumerate(agent_results):
    for j, key in enumerate(DIM_KEYS):
        letter = res.get(key, 'I' if j == 0 else 'N' if j == 1 else 'F' if j == 2 else 'P')
        pred   = DIM_LABEL_MAP[key].get(letter, 0)
        conf   = res.get(f'{key}_conf', 0.5)
        y_pred_axes[i, j] = pred
        y_prob_axes[i, j] = conf if pred == 1 else (1.0 - conf)


# ═══════════════ 4-AXIS METRICS ═══════════════
print('=' * 70)
print('📊  4-AXIS METRICS — RAG + LLM Agent (LangChain)')
print('=' * 70)
print(f'{"Axis":<8} {"F1-Macro":<12} {"Accuracy":<12} {"AUC-ROC":<12}')
print('-' * 44)

f1_list, acc_list, auc_list = [], [], []
for i, name in enumerate(DIM_NAMES):
    f1  = f1_score(y_true_axes[:, i], y_pred_axes[:, i], average='macro', zero_division=0)
    acc = accuracy_score(y_true_axes[:, i], y_pred_axes[:, i])
    try:
        auc = roc_auc_score(y_true_axes[:, i], y_prob_axes[:, i])
    except ValueError:
        auc = 0.5
    f1_list.append(f1); acc_list.append(acc); auc_list.append(auc)
    print(f'{name:<8} {f1:<12.4f} {acc:<12.4f} {auc:<12.4f}')

print('-' * 44)
print(f'{"Average":<8} {np.mean(f1_list):<12.4f} {np.mean(acc_list):<12.4f} '
      f'{np.mean(auc_list):<12.4f}')


# ═══════════════ 16-TYPE METRICS ═══════════════
y_true_type = test_df[LABEL_COL].str.upper().tolist()
y_pred_type = [r['type'].upper() for r in agent_results]

type_to_idx = {t.upper(): i for i, t in enumerate(MBTI_TYPES)}
y_true_16 = np.array([type_to_idx.get(t, 0) for t in y_true_type])
y_pred_16 = np.array([type_to_idx.get(t, 0) for t in y_pred_type])

y_prob_16 = np.zeros((len(agent_results), 16), dtype=float)
for i in range(len(agent_results)):
    for t_idx, t_name in enumerate(MBTI_TYPES):
        t_upper = t_name.upper()
        p = 1.0
        for j, (key, letter) in enumerate(zip(DIM_KEYS, t_upper)):
            target = DIM_LABEL_MAP[key].get(letter, 0)
            p *= y_prob_axes[i, j] if target == 1 else (1.0 - y_prob_axes[i, j])
        y_prob_16[i, t_idx] = p
    row_sum = y_prob_16[i].sum()
    if row_sum > 0:
        y_prob_16[i] /= row_sum

f1_16_macro    = f1_score(y_true_16, y_pred_16, average='macro', zero_division=0)
f1_16_weighted = f1_score(y_true_16, y_pred_16, average='weighted', zero_division=0)
acc_16         = accuracy_score(y_true_16, y_pred_16)

try:
    y_true_16_bin = label_binarize(y_true_16, classes=list(range(16)))
    auc_16 = roc_auc_score(y_true_16_bin, y_prob_16, average='macro', multi_class='ovr')
except Exception:
    auc_16 = 0.0

print(f'\n{"=" * 70}')
print('📊  16-TYPE METRICS — RAG + LLM Agent (LangChain)')
print(f'{"=" * 70}')
print(f'  Accuracy:            {acc_16:.4f}')
print(f'  F1 (Macro):          {f1_16_macro:.4f}')
print(f'  F1 (Weighted):       {f1_16_weighted:.4f}')
print(f'  AUC-ROC (Macro OvR): {auc_16:.4f}')

present_labels = sorted(set(y_true_16.tolist()) | set(y_pred_16.tolist()))
target_names   = [MBTI_TYPES[i].upper() for i in present_labels]
print(f'\n📋 Per-Type Classification Report:')
print(classification_report(
    y_true_16, y_pred_16,
    labels=present_labels,
    target_names=target_names,
    zero_division=0
))


# ═══════════════ SAVE TO CSV ═══════════════
agent_save = pd.DataFrame()
for i, c in enumerate(LABEL_COLS):
    agent_save[f'y_true_{c}'] = y_true_axes[:, i]
    agent_save[f'y_pred_{c}'] = y_pred_axes[:, i]
    agent_save[f'y_prob_{c}'] = np.round(y_prob_axes[:, i], 4)

agent_save['y_true_type'] = y_true_type
agent_save['y_pred_type'] = y_pred_type

agent_save.to_csv(f"{CONFIG['RESULT_DIR']}/rag_agent_predictions.csv", index=False)
print(f'\n✅ Saved → {CONFIG["RESULT_DIR"]}/rag_agent_predictions.csv')